# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hussaintinwala2/Flyrank/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [10]:
!pip -q install duckdb huggingface_hub

import os
import duckdb
from google.colab import userdata

# Get Hugging Face token from Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

# Create DuckDB connection
con = duckdb.connect()

# Authenticate with Hugging Face
con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

# Warehouse location
REL = "hf://datasets/FlyRank/internship-warehouse"

# Tables used in this assignment
TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

print("DuckDB connected successfully.")
print(
    "Daily rows:",
    con.sql(f"SELECT COUNT(*) FROM {TABLES['fact_daily']}").fetchone()[0]
)

print("Connection is working.")

DuckDB connected successfully.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Daily rows: 78835655
Connection is working.


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## 1. My rule and its reason codes

I will build a simple content-review priority score using two pre-decision signals.

**Signal 1 — Staleness:**  
I will use the number of days since `content_updated_date` as a signal of potential refresh need. This is linked to FlyRank's refresh/staleness logic. A longer time since the last update may indicate that a page deserves review, but I will treat this as a prioritization signal rather than proof that the content is outdated.

**Signal 2 — CTR opportunity:**  
I will use observed search CTR (`gsc_clicks / gsc_impressions`) together with `gsc_avg_position`. This is linked to FlyRank's CTR-vs-position logic. A page with search visibility and relatively weak CTR may represent an opportunity for review, but position and query mix can affect CTR, so this is only a directional signal.

The baseline will combine these two signals into one simple score. Each page will receive one reason code and one action label. The score will use only information available at the decision moment and will not use future outcomes or label-derived fields.

### Reason codes

- `STALE_CONTENT` — the page has a long time since its last recorded update.
- `CTR_OPPORTUNITY` — the page has search visibility but observed CTR is weak relative to its search position.
- `REVIEW` — the page meets the baseline priority conditions and should be reviewed.
### Signal checks

**1. Staleness — CONFIRMED**

Staleness is a real, measurable signal in the data. The observed buckets contain 376,200 pages updated within 0–90 days, 34,800 pages at 91–180 days, and 7,047 pages at 181–365 days. This supports using content age/staleness as one signal for prioritizing pages for review or refresh.

**2. CTR — CONFIRMED**

CTR is also a measurable signal with substantial variation across pages. The observed buckets contain 3,413,195 pages below 1% CTR, 133,370 at 1–3%, 28,006 at 3–5%, and 36,490 at 5%+. This supports using low CTR as a signal for search-performance review.

I will combine staleness and CTR into a simple baseline score. The score is intended for decision-support and prioritization, not as proof that a page needs a refresh.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Signal 1: staleness audit
# Use June 2026 as the decision point and calculate days since last update.

staleness = con.sql(f"""
    SELECT
        CASE
            WHEN content_updated_date IS NULL THEN 'MISSING_UPDATE_DATE'
            WHEN DATE_DIFF(
                'day',
                content_updated_date,
                DATE '2026-06-01'
            ) <= 90 THEN '0-90 days'
            WHEN DATE_DIFF(
                'day',
                content_updated_date,
                DATE '2026-06-01'
            ) <= 180 THEN '91-180 days'
            WHEN DATE_DIFF(
                'day',
                content_updated_date,
                DATE '2026-06-01'
            ) <= 365 THEN '181-365 days'
            ELSE '365+ days'
        END AS staleness_bucket,
        COUNT(*) AS n
    FROM {TABLES['dim_content']}
    WHERE is_deleted IS NOT TRUE
    GROUP BY 1
    ORDER BY 1
""").df()

print("Staleness signal:")
print(staleness)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Staleness signal:
  staleness_bucket       n
0        0-90 days  376200
1     181-365 days    7047
2      91-180 days   34800


In [12]:
# Signal 2: CTR opportunity audit
# Use March 2026 as the development window.

ctr_audit = con.sql(f"""
    SELECT
        CASE
            WHEN gsc_impressions = 0 THEN '0 impressions'
            WHEN gsc_clicks / NULLIF(gsc_impressions, 0) < 0.01 THEN '<1% CTR'
            WHEN gsc_clicks / NULLIF(gsc_impressions, 0) < 0.03 THEN '1-3% CTR'
            WHEN gsc_clicks / NULLIF(gsc_impressions, 0) < 0.05 THEN '3-5% CTR'
            ELSE '5%+ CTR'
        END AS ctr_bucket,
        COUNT(*) AS n
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
      AND gsc_impressions > 0
    GROUP BY 1
    ORDER BY 1
""").df()

print("CTR signal:")
print(ctr_audit)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

CTR signal:
  ctr_bucket        n
0   1-3% CTR   133370
1   3-5% CTR    28006
2    5%+ CTR    36490
3    <1% CTR  3413195


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

### Baseline scoring rule

I will score each content page using two observed signals: content staleness and CTR.

The score gives higher priority to pages that are both older and have lower CTR. Staleness contributes up to 50 points and low CTR contributes up to 50 points, giving a maximum score of 100.

Reason codes:
- `STALE_AND_LOW_CTR` — both signals indicate review priority.
- `STALE` — staleness is the stronger signal.
- `LOW_CTR` — low CTR is the stronger signal.
- `NO_STRONG_SIGNAL` — neither signal is strong enough to prioritize.

Action labels:
- `REVIEW_REFRESH` — high-priority page for content review or possible refresh.
- `MONITOR` — lower-priority page that does not currently meet the review threshold.

This is a baseline decision-support rule, not a claim that a page definitely needs a refresh.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Build the baseline ranked queue

import pandas as pd
import os

# Pull the content metadata needed for the rule
baseline = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        content_type,
        word_count,
        char_count,
        search_volume,
        competition,
        content_created_date,
        content_updated_date,
        last_optimized_date,
        is_published,
        is_deleted
    FROM {TABLES['dim_content']}
    WHERE is_published IS TRUE
      AND is_deleted IS NOT TRUE
""").df()

# Calculate staleness in days
baseline["content_updated_date"] = pd.to_datetime(
    baseline["content_updated_date"], errors="coerce"
)

decision_date = pd.Timestamp("2026-03-31")

baseline["days_since_update"] = (
    decision_date - baseline["content_updated_date"]
).dt.days

# Pull March search performance
performance = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
      AND gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").df()

# Calculate CTR
performance["ctr"] = (
    performance["clicks"] /
    performance["impressions"].replace(0, pd.NA)
)

# Join content metadata with performance
baseline = baseline.merge(
    performance,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

# Staleness score: 0–50
baseline["staleness_score"] = (
    baseline["days_since_update"].clip(lower=0, upper=365) / 365 * 50
)

# Low CTR score: 0–50
# CTR below 5% receives a higher score, with <1% receiving the maximum.
baseline["ctr_score"] = (
    ((0.05 - baseline["ctr"]) / 0.05)
    .clip(lower=0, upper=1) * 50
)

# Overall priority score
baseline["priority_score"] = (
    baseline["staleness_score"] +
    baseline["ctr_score"]
)

# Reason code
baseline["reason_code"] = "NO_STRONG_SIGNAL"

baseline.loc[
    (baseline["staleness_score"] >= 25) &
    (baseline["ctr_score"] >= 25),
    "reason_code"
] = "STALE_AND_LOW_CTR"

baseline.loc[
    (baseline["staleness_score"] >= 25) &
    (baseline["ctr_score"] < 25),
    "reason_code"
] = "STALE"

baseline.loc[
    (baseline["staleness_score"] < 25) &
    (baseline["ctr_score"] >= 25),
    "reason_code"
] = "LOW_CTR"

# Action label
baseline["action"] = baseline["priority_score"].apply(
    lambda x: "REVIEW_REFRESH" if x >= 50 else "MONITOR"
)

# Rank the queue
baseline = baseline.sort_values(
    "priority_score",
    ascending=False
).reset_index(drop=True)

baseline["rank"] = baseline.index + 1

# Select useful output columns
queue = baseline[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "priority_score",
        "reason_code",
        "action",
        "days_since_update",
        "ctr",
        "impressions",
        "clicks"
    ]
].copy()

# Create output directory
os.makedirs("work/outputs", exist_ok=True)

# Write the ranked queue
output_path = "work/outputs/baseline_action_score.csv"
queue.to_csv(output_path, index=False)

print(f"Rows in ranked queue: {len(queue):,}")
print(f"CSV written to: {output_path}")

display(queue.head(10))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows in ranked queue: 176,568
CSV written to: work/outputs/baseline_action_score.csv


,rank,client_hash_id,content_hash_id,priority_score,reason_code,action,days_since_update,ctr,impressions,clicks
0,1,client_65de48885f4ef01b,content_19f71daba0876547,91.506849,STALE_AND_LOW_CTR,REVIEW_REFRESH,303,0.0,37.0,0.0
1,2,client_65de48885f4ef01b,content_84106de577dff29c,91.506849,STALE_AND_LOW_CTR,REVIEW_REFRESH,303,0.0,3.0,0.0
2,3,client_65de48885f4ef01b,content_a2e85700106a33b8,91.506849,STALE_AND_LOW_CTR,REVIEW_REFRESH,303,0.0,9.0,0.0
3,4,client_65de48885f4ef01b,content_3ed48fd591b26655,91.506849,STALE_AND_LOW_CTR,REVIEW_REFRESH,303,0.0,1.0,0.0
4,5,client_65de48885f4ef01b,content_ee6f61ff4145746c,91.506849,STALE_AND_LOW_CTR,REVIEW_REFRESH,303,0.0,7.0,0.0
5,6,client_65de48885f4ef01b,content_cdd557d042d8a774,91.506849,STALE_AND_LOW_CTR,REVIEW_REFRESH,303,0.0,19.0,0.0
6,7,client_65de48885f4ef01b,content_38c60323fd1608ec,91.506849,STALE_AND_LOW_CTR,REVIEW_REFRESH,303,0.0,5.0,0.0
7,8,client_65de48885f4ef01b,content_f4685cd9dee88fce,91.506849,STALE_AND_LOW_CTR,REVIEW_REFRESH,303,0.0,1.0,0.0
8,9,client_65de48885f4ef01b,content_26d4238346178145,91.506849,STALE_AND_LOW_CTR,REVIEW_REFRESH,303,0.0,2.0,0.0
9,10,client_65de48885f4ef01b,content_a0fcdebdf75d6f5d,91.506849,STALE_AND_LOW_CTR,REVIEW_REFRESH,303,0.0,1.0,0.0


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

### Top-20 review

The top 20 pages are all assigned `REVIEW_REFRESH` with the reason code `STALE_AND_LOW_CTR`. They score highly because they are approximately 302–303 days since update and have 0% observed CTR.

The main weakness is that several pages have very few impressions. A 0% CTR based on 1–3 impressions is weak evidence, so these recommendations should be treated as decision-support rather than proof that a refresh is needed.

1. **Rank 1:** REVIEW_REFRESH — The page is 303 days stale with 0% CTR. Confidence is low because it has only 37 impressions. The pick could be wrong if the low CTR is caused by very limited search exposure.
2. **Rank 2:** REVIEW_REFRESH — The page is 303 days stale with 0% CTR. Confidence is low because it has only 3 impressions. The pick could be wrong because the CTR estimate is based on very little data.
3. **Rank 3:** REVIEW_REFRESH — The page is 303 days stale with 0% CTR. Confidence is low because it has only 9 impressions. The pick could be wrong if the page simply had insufficient search exposure.
4. **Rank 4:** REVIEW_REFRESH — The page is 303 days stale with 0% CTR. Confidence is very low because it has only 1 impression. The pick could be wrong because one impression is insufficient evidence of poor CTR.
5. **Rank 5:** REVIEW_REFRESH — The page is 303 days stale with 0% CTR. Confidence is low because it has only 7 impressions. The pick could be wrong if the observed CTR is unstable due to low volume.
6. **Rank 6:** REVIEW_REFRESH — The page is 303 days stale with 0% CTR. Confidence is low because it has 19 impressions. The pick could be wrong if the low CTR does not persist with more search exposure.
7. **Rank 7:** REVIEW_REFRESH — The page is 303 days stale with 0% CTR. Confidence is very low because it has only 5 impressions. The pick could be wrong because the sample is small.
8. **Rank 8:** REVIEW_REFRESH — The page is 303 days stale with 0% CTR. Confidence is very low because it has only 1 impression. The pick could be wrong because there is almost no observed search volume.
9. **Rank 9:** REVIEW_REFRESH — The page is 303 days stale with 0% CTR. Confidence is very low because it has only 2 impressions. The pick could be wrong because the CTR signal is based on extremely little data.
10. **Rank 10:** REVIEW_REFRESH — The page is 303 days stale with 0% CTR. Confidence is very low because it has only 1 impression. The pick could be wrong because the observed CTR is not statistically stable.
11. **Rank 11:** REVIEW_REFRESH — The page is 303 days stale with 0% CTR. Confidence is low because it has 9 impressions. The pick could be wrong if more impressions would produce a different CTR.
12. **Rank 12:** REVIEW_REFRESH — The page is 303 days stale with 0% CTR. Confidence is very low because it has only 2 impressions. The pick could be wrong because the evidence is limited.
13. **Rank 13:** REVIEW_REFRESH — The page is 303 days stale with 0% CTR. Confidence is very low because it has only 6 impressions. The pick could be wrong if the low CTR is caused by limited exposure.
14. **Rank 14:** REVIEW_REFRESH — The page is 303 days stale with 0% CTR. Confidence is low because it has 13 impressions. The pick could be wrong if the observed CTR changes with more impressions.
15. **Rank 15:** REVIEW_REFRESH — The page is 303 days stale with 0% CTR. Confidence is very low because it has only 1 impression. The pick could be wrong because there is insufficient evidence of sustained poor CTR.
16. **Rank 16:** REVIEW_REFRESH — The page is 303 days stale with 0% CTR. Confidence is very low because it has only 3 impressions. The pick could be wrong because the CTR signal is based on a tiny sample.
17. **Rank 17:** REVIEW_REFRESH — The page is 303 days stale with 0% CTR. Confidence is very low because it has only 3 impressions. The pick could be wrong because the observed CTR may not represent typical performance.
18. **Rank 18:** REVIEW_REFRESH — The page is 302 days stale with 0% CTR. Confidence is low because it has 15 impressions. The pick could be wrong if the low CTR does not persist over a larger observation window.
19. **Rank 19:** REVIEW_REFRESH — The page is 302 days stale with 0% CTR. Confidence is very low because it has only 2 impressions. The pick could be wrong because the CTR estimate is based on minimal exposure.
20. **Rank 20:** REVIEW_REFRESH — The page is 302 days stale with 0% CTR. Confidence is very low because it has only 3 impressions. The pick could be wrong because the evidence for poor CTR is limited.

**Overall review:** The rule successfully identifies stale pages with low observed CTR, but the top-20 results reveal a limitation: it does not account for impression volume. Therefore, low-volume pages can receive very high scores even when the CTR evidence is weak.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Show the top 20 ranked pages for review

top20 = queue.head(20).copy()

display(
    top20[
        [
            "rank",
            "client_hash_id",
            "content_hash_id",
            "priority_score",
            "reason_code",
            "action",
            "days_since_update",
            "ctr",
            "impressions",
            "clicks"
        ]
    ]
)

,rank,client_hash_id,content_hash_id,priority_score,reason_code,action,days_since_update,ctr,impressions,clicks
0,1,client_65de48885f4ef01b,content_19f71daba0876547,91.506849,STALE_AND_LOW_CTR,REVIEW_REFRESH,303,0.0,37.0,0.0
1,2,client_65de48885f4ef01b,content_84106de577dff29c,91.506849,STALE_AND_LOW_CTR,REVIEW_REFRESH,303,0.0,3.0,0.0
2,3,client_65de48885f4ef01b,content_a2e85700106a33b8,91.506849,STALE_AND_LOW_CTR,REVIEW_REFRESH,303,0.0,9.0,0.0
3,4,client_65de48885f4ef01b,content_3ed48fd591b26655,91.506849,STALE_AND_LOW_CTR,REVIEW_REFRESH,303,0.0,1.0,0.0
4,5,client_65de48885f4ef01b,content_ee6f61ff4145746c,91.506849,STALE_AND_LOW_CTR,REVIEW_REFRESH,303,0.0,7.0,0.0
5,6,client_65de48885f4ef01b,content_cdd557d042d8a774,91.506849,STALE_AND_LOW_CTR,REVIEW_REFRESH,303,0.0,19.0,0.0
6,7,client_65de48885f4ef01b,content_38c60323fd1608ec,91.506849,STALE_AND_LOW_CTR,REVIEW_REFRESH,303,0.0,5.0,0.0
7,8,client_65de48885f4ef01b,content_f4685cd9dee88fce,91.506849,STALE_AND_LOW_CTR,REVIEW_REFRESH,303,0.0,1.0,0.0
8,9,client_65de48885f4ef01b,content_26d4238346178145,91.506849,STALE_AND_LOW_CTR,REVIEW_REFRESH,303,0.0,2.0,0.0
9,10,client_65de48885f4ef01b,content_a0fcdebdf75d6f5d,91.506849,STALE_AND_LOW_CTR,REVIEW_REFRESH,303,0.0,1.0,0.0


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak picks + leakage check

The weakest picks in the top 20 are pages with extremely low impression counts, especially pages with only 1–3 impressions. Although they receive a high score because they are stale and have 0% observed CTR, there is too little search exposure to confidently conclude that their CTR represents sustained poor performance.

This shows a limitation of the baseline rule: it treats 0% CTR similarly regardless of impression volume. A future version could require a minimum impression threshold before treating CTR as a strong signal.

For leakage, the baseline uses only content metadata and March 2026 search-performance observations available at the decision point. It does not use a future outcome, future-window label, `trend_direction`, or `trend_pct`. The output action is therefore based on pre-decision signals rather than a future label.

The rule should be interpreted as a prioritization baseline, not as evidence that every selected page actually needs a refresh.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Leakage and weak-pick checks

print("=== Weak-pick check ===")

weak_picks = queue.head(20).copy()

print("Top-20 pages with <= 3 impressions:")
display(
    weak_picks[weak_picks["impressions"] <= 3][
        [
            "rank",
            "content_hash_id",
            "priority_score",
            "reason_code",
            "action",
            "days_since_update",
            "ctr",
            "impressions",
            "clicks"
        ]
    ]
)

print("\n=== Leakage check ===")

# Confirm that the queue contains only the intended baseline inputs.
baseline_inputs = [
    "days_since_update",
    "ctr",
    "impressions",
    "clicks"
]

print("Baseline inputs used for scoring:")
print(baseline_inputs)

# Check that known future/derived trend fields are not present
leakage_fields = [
    "trend_direction",
    "trend_pct",
    "is_declining",
    "future_clicks",
    "future_impressions"
]

present_leakage_fields = [
    field for field in leakage_fields
    if field in queue.columns
]

print("\nFuture/label-derived fields present in queue:")
print(present_leakage_fields)

if len(present_leakage_fields) == 0:
    print("\nLeakage check PASSED: no known future/label-derived fields are present.")
else:
    print("\nWARNING: potential leakage fields detected.")

=== Weak-pick check ===
Top-20 pages with <= 3 impressions:


,rank,content_hash_id,priority_score,reason_code,action,days_since_update,ctr,impressions,clicks
1,2,content_84106de577dff29c,91.506849,STALE_AND_LOW_CTR,REVIEW_REFRESH,303,0.0,3.0,0.0
3,4,content_3ed48fd591b26655,91.506849,STALE_AND_LOW_CTR,REVIEW_REFRESH,303,0.0,1.0,0.0
7,8,content_f4685cd9dee88fce,91.506849,STALE_AND_LOW_CTR,REVIEW_REFRESH,303,0.0,1.0,0.0
8,9,content_26d4238346178145,91.506849,STALE_AND_LOW_CTR,REVIEW_REFRESH,303,0.0,2.0,0.0
9,10,content_a0fcdebdf75d6f5d,91.506849,STALE_AND_LOW_CTR,REVIEW_REFRESH,303,0.0,1.0,0.0
11,12,content_bbca8d1b7c4a1b86,91.506849,STALE_AND_LOW_CTR,REVIEW_REFRESH,303,0.0,2.0,0.0
14,15,content_325df589607ca257,91.506849,STALE_AND_LOW_CTR,REVIEW_REFRESH,303,0.0,1.0,0.0
15,16,content_d708415e25491756,91.506849,STALE_AND_LOW_CTR,REVIEW_REFRESH,303,0.0,3.0,0.0
16,17,content_4be96fb863be863e,91.506849,STALE_AND_LOW_CTR,REVIEW_REFRESH,303,0.0,3.0,0.0
18,19,content_bdf7d487189e56a8,91.369863,STALE_AND_LOW_CTR,REVIEW_REFRESH,302,0.0,2.0,0.0



=== Leakage check ===
Baseline inputs used for scoring:
['days_since_update', 'ctr', 'impressions', 'clicks']

Future/label-derived fields present in queue:
[]

Leakage check PASSED: no known future/label-derived fields are present.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.